# AZH-521 experiment notebook

This notebook is the researcher-facing entry point for running and inspecting the approved ASTE experiments. It delegates training, checkpoint selection, held-out evaluation, and provenance capture to the existing `aste-run` pipeline; model and evaluator code stay in `src/aste_repro`.

The first execution gate is the 20-run HoASA + EMCGCN baseline (four pinned encoders × seeds `0, 1, 2, 3, 52`). The full protocol is CPU-intensive. The notebook starts no jobs unless an execution flag is explicitly enabled, and it refuses to duplicate a run or overlap the active baseline queue. CASA remains blocked until human-adjudicated, sentiment-bearing ASTE gold is frozen. SSGCN follows the baseline/source gate.

Only manifests marked `complete` with test metrics count as results. Smoke, failed, interrupted, running, and absent runs are never converted into scores.


## 1. Load the frozen study configuration

Run this from the repository root, `sentiment-analysis/`, or this notebook's directory. The matrix JSON is the source of truth for encoder revisions, seeds, targets, and training controls.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import statistics
import subprocess
import time


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs" / "experiment-matrix.json").is_file():
            return candidate
        nested = candidate / "sentiment-analysis"
        if (nested / "configs" / "experiment-matrix.json").is_file():
            return nested
    raise FileNotFoundError("Could not find sentiment-analysis/configs/experiment-matrix.json")


PROJECT = find_project_root()
REPO_ROOT = PROJECT.parent
RUNS = PROJECT / "runs"
REPORT = PROJECT / "results" / "azh-521-report.md"
MATRIX_PATH = PROJECT / "configs" / "experiment-matrix.json"
QUEUE_MARKER = RUNS / "azh-521-hoasa-emcgcn-queue.status.json"
MATRIX = json.loads(MATRIX_PATH.read_text(encoding="utf-8"))
UV = shutil.which("uv")
if UV is None:
    candidates = [Path.home() / ".local" / "bin" / "uv.exe", Path.home() / ".local" / "bin" / "uv"]
    UV = str(next((path for path in candidates if path.exists()), "uv"))

print(f"Project: {PROJECT}")
print(f"Run artifacts: {RUNS}")
print(f"Aggregate report: {REPORT}")
print(f"Declared seeds: {MATRIX['seed_protocol']['seeds']}")
print(f"Training controls: {MATRIX['training']['epochs']} epochs, batch size {MATRIX['training']['batch_size']}")
print(f"Pinned encoders: {', '.join(MATRIX['encoders'])}")


## 2. Inspect run status and recorded test scores

This reads immutable run manifests and metrics written by the training pipeline. It does not evaluate a model or infer scores from logs.


In [ ]:
def read_runs():
    records = []
    if not RUNS.exists():
        return records
    for manifest_path in sorted(RUNS.glob("*/manifest.json")):
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue  # a manifest may be in the middle of being written
        metrics_path = manifest_path.parent / "metrics.json"
        metrics = None
        if metrics_path.is_file():
            try:
                metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                pass
        test_f1 = None
        if manifest.get("status") == "complete" and metrics:
            value = metrics.get("test", {}).get("triplet", {}).get("f1")
            if value is not None:
                test_f1 = 100.0 * float(value)
        records.append({
            "run_id": manifest.get("run_id", manifest_path.parent.name),
            "dataset": manifest.get("dataset"),
            "architecture": manifest.get("architecture"),
            "encoder": manifest.get("encoder"),
            "seed": manifest.get("seed"),
            "status": manifest.get("status"),
            "test_triplet_f1_pct": test_f1,
            "run_dir": str(manifest_path.parent),
        })
    return records


def show_baseline_status():
    rows = [row for row in read_runs() if row["dataset"] == "hoasa" and row["architecture"] == "emcgcn"]
    rows.sort(key=lambda row: (row["encoder"] or "", row["seed"] if row["seed"] is not None else -1))
    if not rows:
        print("No HoASA/EMCGCN run manifests found yet.")
    else:
        for row in rows:
            score = "—" if row["test_triplet_f1_pct"] is None else f"{row['test_triplet_f1_pct']:.2f}%"
            print(f"{row['encoder']:14} seed={str(row['seed']):>2}  {row['status']:12}  test F1={score}  {row['run_id']}")
    return rows


baseline_status = show_baseline_status()
if QUEUE_MARKER.exists():
    queue = json.loads(QUEUE_MARKER.read_text(encoding="utf-8"))
    print(f"\nBaseline queue active (PID {queue.get('queue_pid')}): {queue.get('scope')}")


## 3. Run a selected condition

Set `RUN_ONE_CONDITION = True` only when you intend to launch the selected full benchmark. The function delegates to `aste-run`, preserves the declared seed and source controls, and rejects duplicate complete/running conditions. It also refuses to overlap the active baseline queue.


In [ ]:
def matching_runs(dataset, architecture, encoder, seed):
    return [row for row in read_runs()
            if row["dataset"] == dataset
            and row["architecture"] == architecture
            and row["encoder"] == encoder
            and int(row["seed"]) == int(seed)]


def assert_no_active_queue():
    if QUEUE_MARKER.exists():
        queue = json.loads(QUEUE_MARKER.read_text(encoding="utf-8"))
        raise RuntimeError(
            "An AZH-521 benchmark queue is already active "
            f"(PID {queue.get('queue_pid')}; scope: {queue.get('scope')}). "
            "Inspect run status above; do not start overlapping jobs."
        )


def refresh_report():
    command = [UV, "run", "--python", "3.10", "--project", str(PROJECT),
               "aste-report", "--output", str(REPORT)]
    result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    print(result.stdout.strip())


def run_condition(dataset, architecture, encoder, seed):
    assert_no_active_queue()
    existing = matching_runs(dataset, architecture, encoder, seed)
    if any(row["status"] == "running" for row in existing):
        raise RuntimeError("This condition already has a running benchmark manifest.")
    if any(row["status"] == "complete" for row in existing):
        raise RuntimeError("This seed already has a complete run; refusing to duplicate it.")
    command = [UV, "run", "--python", "3.10", "--project", str(PROJECT),
               "aste-run", "--dataset", dataset, "--architecture", architecture,
               "--encoder", encoder, "--seed", str(seed), "--purpose", "benchmark"]
    print("Launching:", " ".join(command))
    result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise subprocess.CalledProcessError(result.returncode, command, result.stdout, result.stderr)
    refresh_report()
    return result.stdout


RUN_ONE_CONDITION = False  # Change to True before running this cell to launch the selection below.
SELECTED = {"dataset": "hoasa", "architecture": "emcgcn", "encoder": "mbert", "seed": 0}
if RUN_ONE_CONDITION:
    run_condition(**SELECTED)
else:
    print("Execution disabled. Set RUN_ONE_CONDITION = True to launch the selected benchmark.")


## 4. Continue missing HoASA + EMCGCN baseline seeds

This cell is the approved first gate: 20 full runs across four encoders and five seeds. It skips completed seeds, retries failed/interrupted attempts as new immutable runs, and stops if any matching run is already active. Expect multi-day CPU runtime. Keep the execution flag `False` while a queue is active.


In [ ]:
BASELINE_CONDITIONS = [
    {"dataset": "hoasa", "architecture": "emcgcn", "encoder": encoder, "seed": seed}
    for encoder in MATRIX["encoders"]
    for seed in MATRIX["seed_protocol"]["seeds"]
]


def run_missing_baseline():
    assert_no_active_queue()
    for condition in BASELINE_CONDITIONS:
        existing = matching_runs(**condition)
        if any(row["status"] == "running" for row in existing):
            raise RuntimeError(f"Already running: {condition}")
        if any(row["status"] == "complete" for row in existing):
            print(f"Already complete; skip {condition}")
            continue
        run_condition(**condition)


EXECUTE_REMAINING_BASELINE = False  # Change to True only when the queue marker is absent.
if EXECUTE_REMAINING_BASELINE:
    run_missing_baseline()
else:
    print(f"Execution disabled; {len(BASELINE_CONDITIONS)} planned HoASA/EMCGCN seed runs.")


## 5. Seed-level results and descriptive statistics

Only complete runs with recorded held-out test metrics appear here. Scores are exact triplet micro-F1 percentages, matching the aggregate report.


In [ ]:
def baseline_results():
    completed = [row for row in read_runs()
                 if row["dataset"] == "hoasa"
                 and row["architecture"] == "emcgcn"
                 and row["status"] == "complete"
                 and row["test_triplet_f1_pct"] is not None]
    output = []
    for encoder, spec in MATRIX["encoders"].items():
        scores = sorted((row["seed"], row["test_triplet_f1_pct"])
                        for row in completed if row["encoder"] == encoder)
        values = [score for _, score in scores]
        output.append({
            "encoder": spec["display_name"],
            "n_complete": len(values),
            "seed_f1_pct": ", ".join(f"{seed}: {score:.2f}" for seed, score in scores) or "—",
            "mean_pct": round(statistics.mean(values), 2) if values else None,
            "sd_pct": round(statistics.stdev(values), 2) if len(values) > 1 else None,
            "thesis_target_pct": spec["target_mean_triplet_f1"],
        })
    return output


baseline_summary = baseline_results()
for row in baseline_summary:
    print(row)


## 6. Refresh and open the aggregate report

The report retains the full declared matrix, marks blocked/unrun conditions, and computes summaries only from complete run manifests. CASA remains unmeasured until its gold-data gate is resolved; the report must not turn partial or smoke runs into scores.


In [ ]:
refresh_report()
print(f"\nOpen the generated report at: {REPORT}")
